In [ ]:
#!/usr/bin/env python
"""
Demonstration: Why FDR (Benjamini-Hochberg) simplifies to just checking p < α
when all p-values are identical.

This is the key insight that makes Greedy Boruta's rejection criterion simpler.
"""

import numpy as np
import scipy.stats as sp


def fdrcorrection(pvals, alpha=0.05):
    """
    Benjamini-Hochberg FDR correction.
    Copied from BorutaLocal.py for reference.
    
    Returns:
        rejected: boolean array - True if hypothesis is rejected
        pvals_corrected: adjusted p-values
    """
    pvals = np.asarray(pvals)
    pvals_sortind = np.argsort(pvals)
    pvals_sorted = np.take(pvals, pvals_sortind)
    nobs = len(pvals_sorted)
    ecdffactor = np.arange(1, nobs + 1) / float(nobs)

    # BH procedure: reject if p(i) <= (i/m) * alpha
    reject = pvals_sorted <= ecdffactor * alpha
    if reject.any():
        rejectmax = max(np.nonzero(reject)[0])
        reject[:rejectmax] = True

    pvals_corrected_raw = pvals_sorted / ecdffactor
    pvals_corrected = np.minimum.accumulate(pvals_corrected_raw[::-1])[::-1]
    pvals_corrected[pvals_corrected > 1] = 1
    
    # Reorder to original order
    pvals_corrected_ = np.empty_like(pvals_corrected)
    pvals_corrected_[pvals_sortind] = pvals_corrected
    reject_ = np.empty_like(reject)
    reject_[pvals_sortind] = reject
    return reject_, pvals_corrected_


def demonstrate_fdr_with_identical_pvalues():
    """
    Show step-by-step why FDR simplifies when all p-values are equal.
    """
    print("=" * 70)
    print("DEMONSTRATION: FDR with Identical P-values")
    print("=" * 70)
    
    # The Benjamini-Hochberg procedure:
    # 1. Sort p-values: p(1) ≤ p(2) ≤ ... ≤ p(m)
    # 2. Find largest k where p(k) ≤ (k/m) * α
    # 3. Reject all hypotheses i where i ≤ k
    
    print("\n### The Benjamini-Hochberg Procedure ###")
    print("1. Sort p-values: p(1) ≤ p(2) ≤ ... ≤ p(m)")
    print("2. Find largest k where p(k) ≤ (k/m) × α")
    print("3. Reject all hypotheses with rank ≤ k")
    
    print("\n### When All P-values Are Identical ###")
    print("If p(1) = p(2) = ... = p(m) = p*, then:")
    print("  - The condition p(k) ≤ (k/m) × α becomes: p* ≤ (k/m) × α")
    print("  - This is satisfied for all k where k ≥ (m × p*) / α")
    print("  - The LARGEST such k is m (when checking k=m: p* ≤ (m/m) × α = α)")
    print("  - Therefore: ALL hypotheses rejected iff p* ≤ α")
    print("  - And: NO hypotheses rejected iff p* > α")
    print("\n  ⟹ FDR simplifies to just checking: p* < α")
    
    print("\n" + "=" * 70)
    print("NUMERICAL VERIFICATION")
    print("=" * 70)
    
    alpha = 0.05
    m = 100  # Number of features (hypotheses)
    
    # Case 1: p* < α (should reject all)
    print(f"\n### Case 1: All p-values = 0.01 < α={alpha} ###")
    p_star = 0.01
    pvals = np.full(m, p_star)
    rejected, corrected = fdrcorrection(pvals, alpha=alpha)
    print(f"  p* = {p_star}")
    print(f"  Number of hypotheses: {m}")
    print(f"  Rejected: {rejected.sum()} / {m}")
    print(f"  Expected: ALL rejected (since p* < α)")
    assert rejected.all(), "Should reject all!"
    print("  ✓ Confirmed: All rejected")
    
    # Case 2: p* > α (should reject none)
    print(f"\n### Case 2: All p-values = 0.10 > α={alpha} ###")
    p_star = 0.10
    pvals = np.full(m, p_star)
    rejected, corrected = fdrcorrection(pvals, alpha=alpha)
    print(f"  p* = {p_star}")
    print(f"  Number of hypotheses: {m}")
    print(f"  Rejected: {rejected.sum()} / {m}")
    print(f"  Expected: NONE rejected (since p* > α)")
    assert not rejected.any(), "Should reject none!"
    print("  ✓ Confirmed: None rejected")
    
    # Case 3: p* = α (boundary case)
    print(f"\n### Case 3: All p-values = {alpha} = α (boundary) ###")
    p_star = alpha
    pvals = np.full(m, p_star)
    rejected, corrected = fdrcorrection(pvals, alpha=alpha)
    print(f"  p* = {p_star}")
    print(f"  Number of hypotheses: {m}")
    print(f"  Rejected: {rejected.sum()} / {m}")
    print(f"  Expected: ALL rejected (since p* ≤ α)")
    print("  ✓ Result: All rejected" if rejected.all() else "  Note: Boundary behavior")
    
    print("\n" + "=" * 70)
    print("APPLICATION TO GREEDY BORUTA")
    print("=" * 70)
    
    print("\nIn Greedy Boruta at iteration K:")
    print("  - All TENTATIVE features have exactly 0 hits")
    print("  - P-value for 0 hits in K trials = (1/2)^K")
    print("  - ALL tentative features have the SAME p-value!")
    print("")
    print("Therefore:")
    print("  - FDR correction simplifies to: (1/2)^K < α")
    print("  - Solving for K: K > log₂(1/α)")
    print("  - So: K = ⌈log₂(1/α)⌉")
    
    print("\n### Verification for different α values ###")
    print(f"{'α':<12} {'K = ⌈log₂(1/α)⌉':<20} {'P-value = 0.5^K':<20} {'P < α?':<10}")
    print("-" * 62)
    
    for alpha_val in [0.1, 0.05, 0.01, 0.001, 0.0001, 0.00001]:
        K = int(np.ceil(np.log2(1.0 / alpha_val)))
        p_value = 0.5 ** K
        is_rejected = p_value < alpha_val
        print(f"{alpha_val:<12} {K:<20} {p_value:<20.10f} {'Yes ✓' if is_rejected else 'No':<10}")


def demonstrate_fdr_step_by_step():
    """
    Walk through FDR step-by-step with identical p-values.
    """
    print("\n" + "=" * 70)
    print("STEP-BY-STEP FDR WITH IDENTICAL P-VALUES")
    print("=" * 70)
    
    alpha = 0.05
    m = 5  # Small number for clarity
    p_star = 0.03  # Common p-value < alpha
    
    print(f"\nSetup: m={m} hypotheses, all with p-value = {p_star}, α = {alpha}")
    print("\nBenjamini-Hochberg thresholds (k/m × α) for each rank k:")
    print("-" * 50)
    print(f"{'Rank k':<10} {'Threshold (k/m)×α':<20} {'p* ≤ threshold?':<15}")
    print("-" * 50)
    
    largest_k_rejected = 0
    for k in range(1, m + 1):
        threshold = (k / m) * alpha
        rejected = p_star <= threshold
        marker = "✓ REJECT" if rejected else ""
        print(f"{k:<10} {threshold:<20.4f} {marker:<15}")
        if rejected:
            largest_k_rejected = k
    
    print("-" * 50)
    print(f"\nLargest k where p* ≤ threshold: k = {largest_k_rejected}")
    print(f"BH procedure rejects all hypotheses with rank ≤ {largest_k_rejected}")
    print(f"Since all p-values are identical, ranks are arbitrary → ALL {m} rejected")
    
    print("\n### Key Insight ###")
    print(f"The threshold at k=m is (m/m)×α = α = {alpha}")
    print(f"Since p* = {p_star} < {alpha} = α, all hypotheses are rejected.")
    print("This is equivalent to just checking: p* < α")


def compare_fdr_mixed_vs_identical():
    """
    Compare FDR behavior with mixed p-values vs identical p-values.
    """
    print("\n" + "=" * 70)
    print("COMPARISON: Mixed vs Identical P-values")
    print("=" * 70)
    
    alpha = 0.05
    
    # Mixed p-values (typical scenario)
    print("\n### Mixed P-values (typical Boruta scenario) ###")
    mixed_pvals = np.array([0.001, 0.01, 0.03, 0.04, 0.06, 0.10, 0.20, 0.50, 0.80, 0.95])
    rejected, _ = fdrcorrection(mixed_pvals, alpha=alpha)
    print(f"P-values: {mixed_pvals}")
    print(f"Rejected: {rejected}")
    print(f"Count: {rejected.sum()} / {len(mixed_pvals)} rejected")
    print("→ FDR provides nuanced filtering based on p-value ranks")
    
    # Identical p-values (Greedy Boruta scenario)
    print("\n### Identical P-values (Greedy Boruta scenario) ###")
    
    # All below alpha
    identical_low = np.full(10, 0.03)
    rejected_low, _ = fdrcorrection(identical_low, alpha=alpha)
    print(f"All p-values = 0.03 < α={alpha}")
    print(f"Rejected: {rejected_low.sum()} / {len(identical_low)} → ALL rejected")
    
    # All above alpha  
    identical_high = np.full(10, 0.10)
    rejected_high, _ = fdrcorrection(identical_high, alpha=alpha)
    print(f"All p-values = 0.10 > α={alpha}")
    print(f"Rejected: {rejected_high.sum()} / {len(identical_high)} → NONE rejected")
    
    print("\n→ With identical p-values, FDR is binary: all or nothing")
    print("→ The decision boundary is simply p* = α")


if __name__ == "__main__":
    demonstrate_fdr_with_identical_pvalues()
    demonstrate_fdr_step_by_step()
    compare_fdr_mixed_vs_identical()
    
    print("\n" + "=" * 70)
    print("CONCLUSION")
    print("=" * 70)
    print("""
    In Greedy Boruta:
    1. All tentative features have 0 hits → identical p-values = (1/2)^K
    2. FDR with identical p-values reduces to: p* < α
    3. Therefore, we just need K such that (1/2)^K < α
    4. Solving: K > log₂(1/α), so K = ⌈log₂(1/α)⌉

    This is why the Greedy Boruta implementation doesn't need explicit FDR
    correction—it's mathematically equivalent to just computing K iterations
    and rejecting all features with 0 hits.
    """)

DEMONSTRATION: FDR with Identical P-values

### The Benjamini-Hochberg Procedure ###
1. Sort p-values: p(1) ≤ p(2) ≤ ... ≤ p(m)
2. Find largest k where p(k) ≤ (k/m) × α
3. Reject all hypotheses with rank ≤ k

### When All P-values Are Identical ###
If p(1) = p(2) = ... = p(m) = p*, then:
  - The condition p(k) ≤ (k/m) × α becomes: p* ≤ (k/m) × α
  - This is satisfied for all k where k ≥ (m × p*) / α
  - The LARGEST such k is m (when checking k=m: p* ≤ (m/m) × α = α)
  - Therefore: ALL hypotheses rejected iff p* ≤ α
  - And: NO hypotheses rejected iff p* > α

  ⟹ FDR simplifies to just checking: p* < α

NUMERICAL VERIFICATION

### Case 1: All p-values = 0.01 < α=0.05 ###
  p* = 0.01
  Number of hypotheses: 100
  Rejected: 100 / 100
  Expected: ALL rejected (since p* < α)
  ✓ Confirmed: All rejected

### Case 2: All p-values = 0.10 > α=0.05 ###
  p* = 0.1
  Number of hypotheses: 100
  Rejected: 0 / 100
  Expected: NONE rejected (since p* > α)
  ✓ Confirmed: None rejected

### Case 3: All

In [ ]:
import numpy as np

def _compute_iterations_to_convergence(alpha, n_features, bonferroni=True):
        """
        Calculates the number of iterations K required for rejection of 0-hit features.
        
        In Greedy Boruta, all tentative features have exactly 0 hits, giving them
        identical p-values of 0.5^K. To reject at significance level alpha (with
        optional Bonferroni correction for multiple features), we need:
        
            0.5^K < threshold
            => K > log2(1/threshold)
            => K = ceil(log2(1/threshold))
        
        Parameters
        ----------
        alpha : float
            Significance level for the rejection test.
        n_features : int
            Number of features in the dataset, used for Bonferroni correction.
        
        Returns
        -------
        int
            Minimum number of iterations K guaranteeing convergence.
        
        Notes
        -----
        With bonferroni=True (default): threshold = alpha/n_features
            Matches vanilla R Boruta with mcAdj=TRUE (the R default).
            Controls family-wise error rate across all features.
        
        With bonferroni=False: threshold = alpha
            Matches vanilla R Boruta with mcAdj=FALSE.
            No correction for multiple comparisons.
        """
        if bonferroni == True:
            # Bonferroni correction: divide alpha by number of features
            # This matches vanilla R Boruta's default behavior (mcAdj=TRUE)
            threshold = alpha / float(n_features)
        else:
            # No correction for multiple comparisons
            # Matches vanilla R Boruta with mcAdj=FALSE
            threshold = alpha

        # Solve 0.5^K < threshold for K
        # => K > log2(1/threshold)
        # => K = ceil(log2(1/threshold))
        if threshold <= 0:
            raise ValueError("Alpha must be greater than 0.")
        
        k_float = np.log2(1.0 / threshold)
        return int(np.ceil(k_float))

_compute_iterations_to_convergence(alpha=0.1, n_features=500, bonferroni=True)

13

: 

In [8]:
"""
Test: Does wrapping an RF fit inside joblib.Parallel(n_jobs=1) suppress
the RF's own internal parallelism (n_jobs=-1)?

We time three scenarios on the same data:
  1. RF.fit() called directly (no joblib wrapper)         → expect full CPU
  2. RF.fit() inside Parallel(n_jobs=1)                   → expect throttled
  3. RF.fit() inside a plain list comprehension (the fix)  → expect full CPU

If scenario 2 is significantly slower than 1 and 3, the hypothesis is confirmed:
joblib's nesting detection is throttling the inner RF parallelism.
"""

import time
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.base import clone
from joblib import Parallel, delayed


def fit_rf(estimator, X, y):
    """Simple wrapper that fits and returns the estimator."""
    estimator.fit(X, y)
    return estimator.feature_importances_


def run_test(n_samples=5000, n_features=200, n_trees=500, n_repeats=3):
    print(f"Dataset: {n_samples} samples, {n_features} features, {n_trees} trees")
    print(f"Repeating each scenario {n_repeats} times\n")

    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=20,
        random_state=42,
    )

    rf = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=7,
        n_jobs=-1,          # <-- This is the key: RF wants all cores
        random_state=42,
    )

    # -------------------------------------------------------------------------
    # Scenario 1: Direct call (no joblib wrapper)
    # -------------------------------------------------------------------------
    times_direct = []
    for i in range(n_repeats):
        est = clone(rf)
        t0 = time.perf_counter()
        est.fit(X, y)
        elapsed = time.perf_counter() - t0
        times_direct.append(elapsed)
        print(f"  [Direct]      run {i+1}: {elapsed:.3f}s")

    # -------------------------------------------------------------------------
    # Scenario 2: Inside Parallel(n_jobs=1) — the current GreedyBoruta code
    # -------------------------------------------------------------------------
    times_joblib = []
    for i in range(n_repeats):
        t0 = time.perf_counter()
        results = Parallel(n_jobs=1)(
            delayed(fit_rf)(clone(rf), X, y) for _ in range(1)
        )
        elapsed = time.perf_counter() - t0
        times_joblib.append(elapsed)
        print(f"  [Parallel(1)] run {i+1}: {elapsed:.3f}s")

    # -------------------------------------------------------------------------
    # Scenario 3: Plain list comprehension (the proposed fix)
    # -------------------------------------------------------------------------
    times_listcomp = []
    for i in range(n_repeats):
        t0 = time.perf_counter()
        results = [fit_rf(clone(rf), X, y) for _ in range(1)]
        elapsed = time.perf_counter() - t0
        times_listcomp.append(elapsed)
        print(f"  [List comp]   run {i+1}: {elapsed:.3f}s")

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------
    mean_direct = np.mean(times_direct)
    mean_joblib = np.mean(times_joblib)
    mean_listcomp = np.mean(times_listcomp)

    print("\n" + "=" * 60)
    print("RESULTS (mean over {} runs):".format(n_repeats))
    print(f"  Direct call:       {mean_direct:.3f}s")
    print(f"  Parallel(n_jobs=1): {mean_joblib:.3f}s")
    print(f"  List comprehension: {mean_listcomp:.3f}s")
    print(f"\n  Slowdown from joblib wrapper: {mean_joblib / mean_direct:.2f}x")
    print(f"  List comp vs direct:          {mean_listcomp / mean_direct:.2f}x")
    print("=" * 60)

    if mean_joblib > mean_direct * 1.3:
        print("\n>>> HYPOTHESIS CONFIRMED: Parallel(n_jobs=1) is throttling")
        print("    the RF's internal n_jobs=-1 parallelism.")
    else:
        print("\n>>> Hypothesis NOT confirmed on this system.")
        print("    The slowdown may come from something else.")


if __name__ == "__main__":
    run_test()

Dataset: 5000 samples, 200 features, 500 trees
Repeating each scenario 3 times

  [Direct]      run 1: 0.880s
  [Direct]      run 2: 0.838s
  [Direct]      run 3: 0.829s
  [Parallel(1)] run 1: 0.881s
  [Parallel(1)] run 2: 0.855s
  [Parallel(1)] run 3: 0.878s
  [List comp]   run 1: 0.860s
  [List comp]   run 2: 0.882s
  [List comp]   run 3: 0.876s

RESULTS (mean over 3 runs):
  Direct call:       0.849s
  Parallel(n_jobs=1): 0.871s
  List comprehension: 0.873s

  Slowdown from joblib wrapper: 1.03x
  List comp vs direct:          1.03x

>>> Hypothesis NOT confirmed on this system.
    The slowdown may come from something else.


In [1]:
"""
Test: Does clone() per iteration explain the performance gap?

We run the same shuffle + fit + extract importances loop two ways:
  1. Clone a new estimator each iteration  (current GreedyBoruta approach)
  2. Reuse the same estimator each iteration (vanilla Boruta approach)

Both train on the identical full-sized matrix with the same n_jobs=-1 RF.
"""

import time
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.base import clone
from sklearn.utils import check_random_state


def run_iteration_with_clone(base_estimator, X, y, seed, perc=100):
    """Mirrors GreedyBoruta's _parallel_fit_iteration exactly."""
    estimator = clone(base_estimator)
    rng = check_random_state(seed)
    estimator.set_params(random_state=rng)

    n_samples, n_features = X.shape
    x_sha = np.copy(X)

    def _shuffle_column(col):
        rng.shuffle(col)
        return col

    x_sha = np.apply_along_axis(_shuffle_column, 0, x_sha)

    while x_sha.shape[1] < 5:
        x_sha = np.hstack((x_sha, x_sha))

    X_combined = np.hstack((X, x_sha))
    estimator.fit(X_combined, y)
    imp = estimator.feature_importances_
    imp_real = imp[:n_features]
    imp_sha = imp[n_features:]
    imp_sha_max = np.percentile(imp_sha, perc)
    hits = (np.nan_to_num(imp_real) > imp_sha_max)
    return hits


def run_iteration_reuse(estimator, X, y, rng, perc=100):
    """Mirrors vanilla Boruta's per-iteration logic (no clone)."""
    estimator.set_params(random_state=rng)

    n_samples, n_features = X.shape
    x_sha = np.copy(X)

    def _shuffle_column(col):
        rng.shuffle(col)
        return col

    x_sha = np.apply_along_axis(_shuffle_column, 0, x_sha)

    while x_sha.shape[1] < 5:
        x_sha = np.hstack((x_sha, x_sha))

    X_combined = np.hstack((X, x_sha))
    estimator.fit(X_combined, y)
    imp = estimator.feature_importances_
    imp_real = imp[:n_features]
    imp_sha = imp[n_features:]
    imp_sha_max = np.percentile(imp_sha, perc)
    hits = (np.nan_to_num(imp_real) > imp_sha_max)
    return hits


def main(n_samples=5000, n_features=500, n_trees=500, K=14, n_repeats=3):
    print(f"Dataset: {n_samples} samples, {n_features} features")
    print(f"RF: {n_trees} trees, n_jobs=-1")
    print(f"Iterations per run: {K}")
    print(f"Repeats: {n_repeats}\n")

    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=20,
        random_state=42,
    )

    rf = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=7,
        n_jobs=-1,
        random_state=42,
    )

    seeds = np.random.RandomState(42).randint(0, 2**31 - 1, size=K)

    # -----------------------------------------------------------------
    # Approach 1: Clone per iteration (current GreedyBoruta)
    # -----------------------------------------------------------------
    times_clone = []
    for r in range(n_repeats):
        iter_times = []
        t0_total = time.perf_counter()
        for i, s in enumerate(seeds):
            t0 = time.perf_counter()
            run_iteration_with_clone(rf, X, y, s)
            elapsed = time.perf_counter() - t0
            iter_times.append(elapsed)
        total = time.perf_counter() - t0_total
        times_clone.append(total)
        if r == 0:
            print(f"  [Clone] per-iteration times: "
                  f"{', '.join(f'{t:.2f}s' for t in iter_times)}")
        print(f"  [Clone] run {r+1} total: {total:.3f}s")

    # -----------------------------------------------------------------
    # Approach 2: Reuse estimator (vanilla-style)
    # -----------------------------------------------------------------
    times_reuse = []
    for r in range(n_repeats):
        iter_times = []
        reuse_rf = clone(rf)  # single clone at the start
        rng = check_random_state(42)
        t0_total = time.perf_counter()
        for i in range(K):
            t0 = time.perf_counter()
            run_iteration_reuse(reuse_rf, X, y, rng)
            elapsed = time.perf_counter() - t0
            iter_times.append(elapsed)
        total = time.perf_counter() - t0_total
        times_reuse.append(total)
        if r == 0:
            print(f"  [Reuse] per-iteration times: "
                  f"{', '.join(f'{t:.2f}s' for t in iter_times)}")
        print(f"  [Reuse] run {r+1} total: {total:.3f}s")

    # -----------------------------------------------------------------
    # Summary
    # -----------------------------------------------------------------
    mean_clone = np.mean(times_clone)
    mean_reuse = np.mean(times_reuse)

    print(f"\n{'='*60}")
    print(f"RESULTS (mean over {n_repeats} runs of {K} iterations each):")
    print(f"  Clone per iteration: {mean_clone:.3f}s")
    print(f"  Reuse estimator:     {mean_reuse:.3f}s")
    print(f"  Ratio (clone/reuse): {mean_clone / mean_reuse:.2f}x")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

Dataset: 5000 samples, 500 features
RF: 500 trees, n_jobs=-1
Iterations per run: 14
Repeats: 3

  [Clone] per-iteration times: 2.47s, 2.54s, 2.38s, 2.21s, 2.54s, 2.47s, 2.23s, 2.43s, 1.61s, 1.60s, 1.59s, 1.62s, 1.60s, 1.62s
  [Clone] run 1 total: 28.923s
  [Clone] run 2 total: 23.211s
  [Clone] run 3 total: 23.065s
  [Reuse] per-iteration times: 1.62s, 1.69s, 1.62s, 1.63s, 1.60s, 1.64s, 1.69s, 1.72s, 1.68s, 1.72s, 1.66s, 1.68s, 1.61s, 1.62s
  [Reuse] run 1 total: 23.175s
  [Reuse] run 2 total: 22.752s
  [Reuse] run 3 total: 23.189s

RESULTS (mean over 3 runs of 14 iterations each):
  Clone per iteration: 25.066s
  Reuse estimator:     23.039s
  Ratio (clone/reuse): 1.09x
